# RFM Customer Segmentation - EDA
### Based on: Wong, Tong & Haw (2024) - Exploring Customer Segmentation in E-Commerce using RFM Analysis with Clustering Techniques
### Journal: Journal of Telecommunications and the Digital Economy, Vol. 12 No. 3, pp. 97–125
### DOI: https://doi.org/10.18080/jtde.v12n3.978

In [0]:
from pyspark.sql.functions import col, count, when, avg, stddev
from pyspark.sql.functions import min as spark_min, max as spark_max
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

COLOR_MAIN    = "#1B3A6B"
COLOR_WARN    = "#E65C00"
COLOR_OK      = "#00695C"
COLOR_BAD     = "#B71C1C"
COLOR_NEUTRAL = "#9E9E9E"

print("imports done")

**Load Feature Table**

In [0]:
churn_features = spark.table("bharatmart.ml.churn_features")

print(f"rows : {churn_features.count():,}")

In [0]:
display(churn_features.limit(5))

all 16 columns are present. 92,107 customers, one row per customer.

the three RFM columns we need for model 2 are here:
- recency_days - days since last order (R)
- total_orders - total order count per customer (F)
- total_spent - total amount spent per customer (M)

these two were excluded from model 1 to prevent leakage because they 
were part of the churn label formula. for model 2 there is no label - 
clustering is unsupervised. using them here is correct and matches 
Wong et al. (2024) exactly.

CUST0078637 has total_spent = 97,092 on only 8 orders - clear outlier.
avg_order_value of 12,136 is far outside normal range.
we will clip total_spent at p99 before clustering to handle this.

**RFM Feature Selection**

In [0]:
rfm_df = churn_features.select("customer_id", "recency_days", "total_orders", "total_spent")

print(f"customers : {rfm_df.count():,}")
print(f"null recency : {rfm_df.filter(col('recency_days').isNull()).count()}")
print(f"null orders : {rfm_df.filter(col('total_orders').isNull()).count()}")
print(f"null spent : {rfm_df.filter(col('total_spent').isNull()).count()}")

92,107 customers. zero nulls across all three RFM columns.

no imputation needed before clustering. the data is clean and ready.
Wong et al. (2024) removed records with missing customer IDs before 
computing RFM values. model 1 feature engineering already did this - 
32,284 ghost orders were dropped at that stage. we inherit clean data here.

**RFM Descriptive Statistics**

In [0]:
stats = rfm_df.select(
    spark_min("recency_days").alias("R_min"),
    spark_max("recency_days").alias("R_max"),
    avg("recency_days").alias("R_mean"),
    stddev("recency_days").alias("R_std"),
    spark_min("total_orders").alias("F_min"),
    spark_max("total_orders").alias("F_max"),
    avg("total_orders").alias("F_mean"),
    stddev("total_orders").alias("F_std"),
    spark_min("total_spent").alias("M_min"),
    spark_max("total_spent").alias("M_max"),
    avg("total_spent").alias("M_mean"),
    stddev("total_spent").alias("M_std"),
).toPandas()

print(stats.T.to_string())

three things stand out from these numbers.

recency_days: mean is 81 days but std is 174. the max is 2,258 days - 
a customer who last ordered over 6 years ago. the std is more than 
double the mean which means the distribution is heavily right skewed.

total_orders: mean 17, max 54, std 9.8. this is the most well-behaved 
of the three. still right skewed but not extreme.

total_spent: mean is 20,548 but max is 827,799. std of 60,339 is 3x 
the mean. this is the most skewed of all three. the 827,799 value is 
almost certainly the same outlier we saw earlier - CUST0078637 with 
avg_order_value of 12,136. we will clip total_spent at p99.

all three distributions are right skewed. this is exactly what 
Wong et al. (2024) found in their e-commerce dataset. they tested 
log transformation and Power Transformer with Yeo-Johnson to fix this 
skewness before clustering. Yeo-Johnson won because it handles both 
positive and negative skew, and works on zero values - log transform 
fails where recency_days could be 0. we will apply the same approach.

**Recency Distribution**

In [0]:
recency_pdf = rfm_df.select("recency_days").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(recency_pdf["recency_days"].dropna(), bins=50, 
        color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Recency Days")
ax.set_ylabel("Customer Count")
ax.set_title("Recency Distribution — Days Since Last Order")
plt.tight_layout()
plt.show()

the chart confirms what the stats showed - extreme right skew.

roughly 45,000 customers ordered within the last 0-50 days. 
these are the active buyers. the count drops sharply after that.

the long flat tail stretches all the way to 2,258 days - customers 
who last ordered over 6 years ago and never came back. very few 
customers sit in the 500-2000 day range which makes sense. 
either you come back or you don't.

this skew is a problem for clustering. K-Means uses euclidean distance - 
a customer at day 2,258 will be treated as astronomically far from 
everyone else. it will pull cluster centroids toward it and distort 
the entire segmentation.

Wong et al. (2024) saw the same pattern in their dataset. log 
transformation helped but did not fully fix it. Power Transformer 
with Yeo-Johnson reduced the skew more completely because it handles 
the long tail without breaking on the zero values we see on the left.

we will apply Yeo-Johnson in notebook 10b before running any clustering.

**Frequency Distribution**

In [0]:
orders_pdf = rfm_df.select("total_orders").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(orders_pdf["total_orders"].dropna(), bins=50,
        color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Total Orders per Customer")
ax.set_ylabel("Customer Count")
ax.set_title("Frequency Distribution — Total Orders per Customer")
plt.tight_layout()
plt.show()

the distribution is right skewed but not as extreme as recency.

most customers sit between 1 and 30 orders. the count gradually 
rises from 1 to about 8-10 orders, stays relatively flat, then 
drops off after 30.

there is one sharp spike at 19 orders - nearly 5,800 customers 
land exactly at 19. this is unusual. a natural distribution would 
not produce a spike this sharp at one specific value. it could be 
a loyalty program milestone, a data generation pattern, or a 
platform incentive that triggered a burst of orders at that number. 
worth noting but not a blocker for clustering.

the tail goes to 54 orders max. customers beyond 40 orders are very 
few - those are the most loyal buyers on the platform.

compared to recency, frequency is better behaved. the skew is milder. 
but Yeo-Johnson will still help here - the spike at 19 and the right 
tail will be smoothed before clustering, which stops that one value 
from pulling a cluster centroid toward it.

**Monetary Distribution**

In [0]:
spent_pdf = rfm_df.select("total_spent").toPandas()
spent_pdf["total_spent"] = spent_pdf["total_spent"].astype(float)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(spent_pdf["total_spent"].dropna(), bins=50,
        color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Total Spent (Rs.)")
ax.set_ylabel("Customer Count")
ax.set_title("Monetary Distribution — Total Spent per Customer (Raw)")
plt.tight_layout()
plt.show()

same problem we saw in model 1 with raw order amounts.

almost all 92,107 customers are squeezed into the first bar. 
the x-axis stretches to 827,799 because of a handful of extreme 
outliers - the chart is unreadable.

79,000+ customers spent under 50,000 Rs. total. the few customers 
beyond 200,000 are almost certainly B2B buyers or data anomalies - 
not typical BharatMart shoppers.

we need to clip at p99 to see the real distribution.

**Monetary Distribution - Clipped at p99**

In [0]:
p99_spent = float(np.percentile(spent_pdf["total_spent"].dropna(), 99))
print(f"p99 total_spent = Rs. {p99_spent:,.0f}")

clipped = spent_pdf[spent_pdf["total_spent"] <= p99_spent]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clipped["total_spent"], bins=50, color=COLOR_OK, edgecolor="white")
ax.set_xlabel(f"Total Spent (Rs.) — clipped at p99")
ax.set_title(f"Monetary Distribution — clipped at p99 (Rs. {p99_spent:,.0f})")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

p99 is Rs. 307,047 but the chart still looks the same - 78,000 
customers squeezed into the first bar. clipping at p99 removed 
the extreme outliers above 307,047 but the remaining data is 
still too skewed to read.

this tells us the real spending of most customers is very low - 
probably under 10,000 Rs. lifetime. the p99 of 307,047 means 
even the top 1% threshold is very high, which shows just how 
concentrated the bottom 99% are on the left.

we need to clip tighter - at p95 - to actually see the shape 
of where most customers sit.

**Monetary Distribution - Clipped at p95**

In [0]:
p95_spent = float(np.percentile(spent_pdf["total_spent"].dropna(), 95))
p50_spent = float(np.percentile(spent_pdf["total_spent"].dropna(), 50))
print(f"p50 total_spent = Rs. {p50_spent:,.0f}")
print(f"p95 total_spent = Rs. {p95_spent:,.0f}")
print(f"p99 total_spent = Rs. {p99_spent:,.0f}")

clipped95 = spent_pdf[spent_pdf["total_spent"] <= p95_spent]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clipped95["total_spent"], bins=50, color=COLOR_OK, edgecolor="white")
ax.set_xlabel(f"Total Spent (Rs.) — clipped at p95")
ax.set_title(f"Monetary Distribution — clipped at p95 (Rs. {p95_spent:,.0f})")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

p50 is Rs. 1,854 - half of all 92,107 customers spent less than 
Rs. 1,854 lifetime. that is the real story here.

the mean was 20,548 but the median is only 1,854. that gap tells 
you how hard a few high-value customers are pulling the average up.

even at p95 the chart is still unreadable - 67,000 customers are 
in the first bar under 5,000 Rs. the distribution is so concentrated 
at the low end that clipping alone won't help visualise it.

we need to zoom into the 0 to 10,000 range where the bulk of 
customers actually sit.

**Monetary Distribution - Zoomed 0 to 10,000**

In [0]:
zoomed = spent_pdf[spent_pdf["total_spent"] <= 10000]

print(f"customers under Rs.10,000 : {len(zoomed):,}  ({len(zoomed)/len(spent_pdf)*100:.1f}%)")
print(f"customers above Rs.10,000 : {len(spent_pdf) - len(zoomed):,}  ({(len(spent_pdf)-len(zoomed))/len(spent_pdf)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(zoomed["total_spent"], bins=50, color=COLOR_OK, edgecolor="white")
ax.axvline(x=p50_spent, color=COLOR_BAD, linestyle="--", linewidth=1.5, label=f"median Rs. {p50_spent:,.0f}")
ax.set_xlabel("Total Spent (Rs.) — zoomed 0 to 10,000")
ax.set_ylabel("Customer Count")
ax.set_title("Monetary Distribution — Zoomed into 0–10,000 Rs.")
ax.legend()
plt.tight_layout()
plt.show()

now we can see the real shape.

85.7% of customers - 78,947 out of 92,107 - spent under Rs. 10,000 
lifetime. the remaining 14.3% account for everything above that.

the distribution peaks around Rs. 400-1,800 then drops off steadily. 
the median at Rs. 1,854 sits right at the peak - half the customer 
base spent less than this amount across their entire relationship 
with BharatMart.

this is a classic long-tail spending pattern. a small group of 
high-value customers is doing a disproportionate amount of the 
spending. those are the Champions. the bulk of customers are 
low-to-mid spenders - they are the Potential Loyalists and At Risk 
segments we expect to find after clustering.

total_spent is the most skewed of all three RFM features. Yeo-Johnson 
will do the most work on this column. without normalisation, K-Means 
would treat the distance between Rs. 500 and Rs. 50,000 as massive 
compared to a recency difference of 10 days - and cluster purely on 
spend, ignoring recency and frequency entirely.

Wong et al. (2024) found the same in their dataset - monetary value 
had the most extreme skew and benefited most from Power Transformer 
normalisation before clustering.

we will clip total_spent at p99 = Rs. 307,047 in notebook 10b 
before applying Yeo-Johnson.

**Skewness Check**

In [0]:
r_skew = float(recency_pdf["recency_days"].skew())
f_skew = float(orders_pdf["total_orders"].skew())
m_skew = float(spent_pdf["total_spent"].skew())

print(f"recency_days skewness : {r_skew:.2f}")
print(f"total_orders skewness : {f_skew:.2f}")
print(f"total_spent skewness : {m_skew:.2f}")
print()
print("rule of thumb: skew > 1.0 = significant right skew")
print("skew > 2.0 = severe skew, normalisation critical")

these three numbers tell us exactly why normalisation is non-negotiable.

recency_days skew = 8.56 - severe. the most skewed of all three.
the 2,258-day outliers we saw in the histogram are driving this.
without normalisation K-Means would cluster almost entirely on 
recency and ignore the other two dimensions.

total_spent skew = 4.28 - severe. confirmed by what we saw in the 
zoomed chart. 85.7% of customers under Rs. 10,000 with a long tail 
stretching to Rs. 827,799.

total_orders skew = 0.36 - mild. the most well-behaved of the three. 
the spike at 19 orders adds a little skew but nothing severe.

Wong et al. (2024) tested two normalisation methods on their dataset 
before clustering. log transformation reduced skew but failed to 
fully fix it. Power Transformer with Yeo-Johnson achieved a more 
symmetrical distribution across all three RFM features.

log transform would also fail here on recency_days = 0 since 
log(0) is undefined. we have customers with recency_days = 0 
meaning they ordered today. Yeo-Johnson handles zero and negative 
values without breaking - that alone makes it the correct choice 
for this dataset.

we will apply PowerTransformer(method='yeo-johnson') in notebook 10b.

**RFM Scatter Plots**

In [0]:
rfm_pdf = rfm_df.toPandas()
rfm_pdf["total_orders"] = rfm_pdf["total_orders"].astype(float)
rfm_pdf["total_spent"]  = rfm_pdf["total_spent"].astype(float)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(rfm_pdf["recency_days"], rfm_pdf["total_orders"],
                alpha=0.1, s=5, color=COLOR_MAIN)
axes[0].set_xlabel("Recency Days (R)")
axes[0].set_ylabel("Total Orders (F)")
axes[0].set_title("R vs F")

axes[1].scatter(rfm_pdf["total_orders"], rfm_pdf["total_spent"],
                alpha=0.1, s=5, color=COLOR_OK)
axes[1].set_xlabel("Total Orders (F)")
axes[1].set_ylabel("Total Spent (M)")
axes[1].set_title("F vs M")

axes[2].scatter(rfm_pdf["recency_days"], rfm_pdf["total_spent"],
                alpha=0.1, s=5, color=COLOR_WARN)
axes[2].set_xlabel("Recency Days (R)")
axes[2].set_ylabel("Total Spent (M)")
axes[2].set_title("R vs M")

plt.tight_layout()
plt.show()

all three plots show the same problem from different angles - 
the raw data is too skewed to cluster meaningfully.

R vs F (left): all customers are compressed against the left wall. 
the dense blue mass is 0-200 recency days. everything beyond 500 
days is a thin scatter of dots. K-Means would draw cluster boundaries 
through that dense wall and produce meaningless segments.

F vs M (middle): the vertical striping is the spike at 19 orders 
we saw earlier - a thick column of customers all at exactly that 
value. the M axis stretches to 800,000 because of outliers. the 
actual customer mass is sitting flat along the bottom, invisible.

R vs M (right): almost everything is compressed into a thin orange 
line against the y-axis. the outlier customers above 200,000 Rs. 
are the only visible points. the 85.7% of customers under Rs. 10,000 
are all stacked on top of each other at the bottom.

there are no natural visual clusters here. not because the segments 
don't exist, they do but because the scale differences between 
dimensions are too large. a recency gap of 100 days looks tiny 
next to a spend gap of 100,000 Rs. K-Means treats both as euclidean 
distance and spend completely dominates.

this is exactly the problem Wong et al. (2024) solved with 
Power Transformer normalisation. after Yeo-Johnson these three 
plots will show a much more balanced spread and the cluster 
boundaries will be driven by all three dimensions equally.

**Correlation Between RFM Features**

In [0]:
corr_matrix = rfm_pdf[["recency_days", "total_orders", "total_spent"]].corr()

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(["recency", "orders", "spent"], rotation=45)
ax.set_yticklabels(["recency", "orders", "spent"])

for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", color="white", fontsize=12)

ax.set_title("RFM Feature Correlation")
plt.tight_layout()
plt.show()

three nearly independent features. that's the good news here.

recency vs orders: -0.15. customers who ordered recently tend to 
have slightly more orders. weak but makes sense.

recency vs spent: -0.14. same story. recent customers spent a 
bit more. again, weak.

orders vs spent: 0.03. basically zero. how many times someone 
ordered tells you almost nothing about how much they spent total. 
a customer with 40 orders could be spending Rs. 200 each time. 
another with 10 orders could have spent Rs. 50,000.

for clustering this is good. when features are highly correlated 
one of them is just a duplicate, it adds noise without adding 
information. here all three are measuring something genuinely 
different about each customer.

Wong et al. (2024) used the same three RFM features without 
dropping any due to correlation. our numbers confirm that was 
the right call for BharatMart too.

**EDA Summary**

In [0]:
print("=" * 50)
print("RFM EDA SUMMARY — inputs for 10b_rfm_clustering")
print("=" * 50)
print(f"\ncustomers : 92,107")
print(f"nulls : 0 across all 3 features")
print(f"\nskewness:")
print(f"recency_days : {r_skew:.2f} — severe")
print(f"total_orders : {f_skew:.2f} — mild")
print(f"total_spent : {m_skew:.2f} — severe")
print(f"\noutlier clip:")
print(f"total_spent p99 : Rs. {p99_spent:,.0f}")
print(f"total_spent p50 : Rs. {p50_spent:,.0f}")
print(f"\ncorrelation: all 3 features independent (max 0.15)")
print(f"\nnormalisation : PowerTransformer(yeo-johnson)")
print(f"reason : log fails on recency_days = 0")
print(f"yeo-johnson handles zero values")
print(f"matches Wong et al. (2024) finding")
print(f"\nalgorithms to run : KMeans + Hierarchical Agglomerative")
print(f"k selection : Elbow Method k=2 to 10")
print(f"evaluation : Silhouette Score + Calinski-Harabasz")
print("=" * 50)